In [10]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import genextreme as gev

from unseen import fileio
from unseen import process_utils
from unseen import eva
from unseen import stability
from unseen import independence
from unseen import bias_correction
from unseen import time_utils

In [ ]:
var = 'tasmax'

In [18]:
def likelihood(data, event):
    """Calculate event likelihood"""

    shape, loc, scale = eva.fit_gev(data)
    probability = gev.sf(event, shape, loc=loc, scale=scale)
    return_period = 1. / probability
    percentile = (1 - probability) * 100
    print(f'{return_period:.0f} year return period')
    print(f'{percentile:.2f}% percentile')

    return shape, loc, scale

## Observations

In [2]:
obs_file = '/g/data/xv83/unseen-projects/outputs/wcrp-txx/data/txx_ERA5_1940-2025_annual_PNW.nc'

In [3]:
obs_ds = fileio.open_dataset(obs_file)
obs_ds = obs_ds.dropna("time")

In [6]:
obs_max_event = obs_ds["tasmax"].values.max()
print(obs_max_event)

39.51474681653524


TODO: Use `time_utils.select_time_period` to select the period of time used in the bias correction and similarity testing (i.e. 1971-2018). (Check if there is also a detrending that happens in the bias correction and similarity testing)

## Model

In [ ]:
model_ds = fileio.open_dataset('/g/data/xv83/unseen-projects/outputs/wcrp-txx/data/txx_CanESM5-dcppA-hindcast_196101-201701_annual_PNW.nc')

In [ ]:
model_ds = model_ds.compute()
model_ds

#### Stability and stationarity

In [ ]:
stability.create_plot(
    model_ds[var],
    'TXx',
    [1960, 1970, 1980, 1990, 2000, 2010],
    uncertainty=False,
    return_method='gev',
    units='TXx (degC)',
    ylim=(25, 50),
)

#### Independence

In [ ]:
mean_correlations, null_correlation_bounds = independence.run_tests(model_ds[var])

In [ ]:
independence.create_plot(
    mean_correlations,
    null_correlation_bounds,
    'independence.png'
)

In [ ]:
min_lead = 0

Drop lead times that aren't independent...

In [ ]:
model_da_indep = model_ds[var].where(model_ds['lead_time'] > min_lead)
model_da_indep = model_da_indep.dropna('lead_time')

#### Bias correction

In [ ]:
correction_method = 'additive'
baseline_period = ['1970-01-01', '2018-12-30']

In [ ]:
bias = bias_correction.get_bias(
    model_da_indep,
    obs_ds[var],
    correction_method,
    time_rounding='A',
    time_period=baseline_period
)
print(bias)

In [ ]:
model_da_bc = bias_correction.remove_bias(model_da_indep, bias, correction_method)

In [ ]:
model_da_bc = model_da_bc.compute()

#### Similarity testing

In [ ]:
model_da_indep_stacked = model_da_indep.stack({'sample': ['ensemble', 'init_date', 'lead_time']})

In [ ]:
model_da_bc_stacked = model_da_bc.dropna('lead_time').stack({'sample': ['ensemble', 'init_date', 'lead_time']})

In [ ]:
model_da_indep_stacked

In [ ]:
fig, ax = plt.subplots(figsize=[8, 5])
xvals = np.arange(25, 40, 0.1)

model_da_indep.plot.hist(bins=20, density=True, alpha=0.7, facecolor='tab:blue')
model_raw_shape, model_raw_loc, model_raw_scale = eva.fit_gev(model_da_indep_stacked.values, fitstart='scipy_subet')
model_raw_pdf = gev.pdf(xvals, model_raw_shape, model_raw_loc, model_raw_scale)
plt.plot(xvals, model_raw_pdf, color='tab:blue', linewidth=4.0, label='model')

model_da_bc.plot.hist(bins=50, density=True, alpha=0.7, facecolor='tab:orange')
model_bc_shape, model_bc_loc, model_bc_scale = eva.fit_gev(model_da_bc_stacked.values, fitstart='scipy_subet')
model_bc_pdf = gev.pdf(xvals, model_bc_shape, model_bc_loc, model_bc_scale)
plt.plot(xvals, model_bc_pdf, color='tab:orange', linewidth=4.0, label='model (corrected)')

obs_ds[var].plot.hist(ax=ax, bins=50, density=True, facecolor='tab:gray', alpha=0.7)
plt.plot(xvals, agcd_pdf, color='tab:gray', linewidth=4.0, label='observations')

plt.xlabel('Rx5day (mm)')
plt.ylabel('probability')
plt.title('Hobart')
plt.xlim(0, 250)
plt.legend()
plt.grid()
plt.savefig(
    'distribution.png',
    bbox_inches='tight',
    facecolor='white',
    dpi=200
)
plt.show()

In [ ]:
moments.create_plot(
    model_da_indep,
    obs_ds[var],
    da_bc_fcst=model_da_bc,
)

In [ ]:
similarity_ds = similarity.similarity_tests(model_da_indep, obs_ds[var], var)
print('KS score:', similarity_ds['ks_statistic'].values)
print('KS p-value:', similarity_ds['ks_pval'].values)
print('AD score:', similarity_ds['ad_statistic'].values)
print('AD p-value:', similarity_ds['ad_pval'].values)

In [ ]:
similarity_bc_ds = similarity.similarity_tests(model_da_bc, obs_ds[var], var)
print('KS score:', similarity_bc_ds['ks_statistic'].values)
print('KS p-value:', similarity_bc_ds['ks_pval'].values)
print('AD score:', similarity_bc_ds['ad_statistic'].values)
print('AD p-value:', similarity_bc_ds['ad_pval'].values)

#### Process assessment

In [ ]:
model_da_indep_stacked_df = model_da_indep_stacked.to_dataframe()

In [ ]:
model_da_indep_stacked_df.sort_values(by=[var], ascending=False)

In [ ]:
process_utils.plot_event_seasonality(model_da_indep_stacked_df)

In [ ]:
process_utils.plot_circulation(
    model_da_indep_stacked_df,
    event_var=var,
    top_n_events=3,
    event_duration=5,
    infile_list='/home/599/dbi599/unseen-projects/file_lists/CanESM5_dcppA-hindcast_tasmax_files.txt',
    color_var=var,
    contour_var='psl',
    color_levels=[0, 30, 60, 90, 120, 150, 180, 210],
    init_year_offset=0,
)

## Results

In [ ]:
likelihood(model_da_indep_stacked.values, obs_max_event)

In [ ]:
fig = plt.figure(figsize=[6, 4])
ax = fig.add_subplot()
eva.plot_gev_return_curve(
    ax,
    model_da_bc_stacked.values,
    rx5day_max,
    n_bootstraps=1000,
    direction="exceedance",
    ylabel='Rx5day (mm)',
    ylim=(0, 400),
)
plt.savefig(
    'return_curve.png',
    bbox_inches='tight',
    facecolor='white',
    dpi=200
)